# Least and greatest waterfall-equilibrium prices

This notebook reproduces the computational examples in *Asset Value and Securitization under Heterogeneous Beliefs*. It computes only the least equilibrium price $q_{\mathcal T}^{\min}$ and the greatest equilibrium price $q_{\mathcal T}^{\max}$, using the monotone iterations in Proposition 1. It does not search for intermediate equilibria.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repository_url = (
    "https://github.com/oliverpardo1979/"
    "Asset-value-and-securitization-under-heterogeneous-beliefs-anew.git"
)
repository_dir = Path("/content/asset-value-securitization")

if repository_dir.exists():
    subprocess.run(
        ["git", "-C", str(repository_dir), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", repository_url, str(repository_dir)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repository_dir / "computational_companion" / "requirements.txt"),
    ],
    check=True,
)
os.chdir(repository_dir)
print(f"Repository ready at {repository_dir}")

## Paper examples

The four cases below compare the no-tranching and tranching versions of the motivating example and the reviewer's multiplicity example.

In [ ]:
import numpy as np
from IPython.display import Markdown, display

from computational_companion.paper_examples import all_paper_examples
from computational_companion import compute_extreme_equilibria


def format_vector(values):
    return "(" + ", ".join(f"{value:.3f}" for value in values) + ")"


rows = []
for example in all_paper_examples():
    result = compute_extreme_equilibria(example.economy)
    rows.append(
        (
            example.name,
            format_vector(result.q_min),
            format_vector(result.q_max),
            "Yes" if result.multiplicity_detected() else "No",
            max(result.residual_min, result.residual_max),
        )
    )

table = [
    r"| Case | $q^{\min}$ | $q^{\max}$ | Multiplicity | Maximum residual |",
    "| --- | --- | --- | --- | --- |",
]
table.extend(
    f"| {name} | {q_min} | {q_max} | {multiple} | {residual:.2e} |"
    for name, q_min, q_max, multiple, residual in rows
)
display(Markdown("\n".join(table)))

The least and greatest prices coincide in the two motivating cases and in the reviewer's no-tranching benchmark. They differ in the reviewer's three-tranche case, which establishes multiplicity.

## Verification

The regression tests compare the computed extreme equilibria with the values reported in the paper. They also verify directly that the reported intermediate price vector in the reviewer's example is a fixed point, although the solver does not return it.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "computational_companion.test_waterfall_equilibria",
    ],
    cwd=repository_dir,
    check=True,
)